In [17]:
import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\acer\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\acer\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\acer\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [18]:
df = pd.read_csv('../data/raw/FR_Dataset.csv')

In [19]:
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def clean_text(text):
    # 1 lower the text
    text = text.lower()

    # 2 remove punctuation/special characters
    text = re.sub(r'[^a-zA-Z\s]','',text)

    # 3 split into words(tokenization)
    words = text.split()
    
    # 4. remove stopwords AND lemmatize each remaining word
    words = [lemmatizer.lemmatize(word) for word in words if word not in stop_words]

    # 5. join back into a single string
    return ' '.join(words)

In [20]:
sample = df['text_'].iloc[0]
print("Before:",sample)
print("After:",clean_text(sample))

Before: Love this!  Well made, sturdy, and very comfortable.  I love it!Very pretty
After: love well made sturdy comfortable love itvery pretty


In [21]:
df['clean_text'] = df['text_'].apply(clean_text)


In [22]:
df[['text_' ,'clean_text']].sample(5)

,text_,clean_text
12788,"Good movie, if you ever saw it, you will love ...",good movie ever saw love one love itvery good
39718,These are beautiful navy pumps! They are ligh...,beautiful navy pump lightweight comfortable lo...
29026,This book came into my life at exactly the rig...,book came life exactly right moment already re...
37664,"These shoes are the most comfortable shoes, I ...",shoe comfortable shoe would buy another pair f...
10670,3 years later these are still my best online p...,year later still best online purchase ever rep...


In [23]:
df.to_csv('../data/processed/cleaned_reviews.csv', index=False)

Feature 1: Review length

In [24]:
df['review_length'] = df['text_'].apply(len)

Feature 2: Sentiment score using TextBlob

In [25]:
pip install textblob

Note: you may need to restart the kernel to use updated packages.


In [26]:
from textblob import TextBlob
df['sentiment'] = df['text_'].apply(lambda x: TextBlob(x).sentiment.polarity)

Feature 3: Sentiment-rating mismatch

In [27]:
df['scaled_rating'] = -1 + (df['rating'] - 1) * 0.5

df['sentiment_mismatch'] = abs(df['scaled_rating'] - df['sentiment'])

In [28]:
df[['rating', 'sentiment', 'scaled_rating', 'sentiment_mismatch']].sample(5)

,rating,sentiment,scaled_rating,sentiment_mismatch
3589,5.0,0.247101,1.0,0.752899
18085,5.0,0.162500,1.0,0.837500
34989,3.0,-0.200000,0.0,0.200000
37146,4.0,0.425000,0.5,0.075000
14809,2.0,0.140351,-0.5,0.640351


In [29]:
df['exclamation_count'] = df['text_'].apply(lambda x: x.count('!'))

In [30]:
superlatives = ['amazing', 'perfect', 'best', 'excellent', 'awesome', 'incredible', 'love', 'great']

def count_superlatives(text):
    words = text.split()
    return sum(1 for word in words if word in superlatives)

df['superlative_count'] = df['clean_text'].apply(count_superlatives)

In [31]:
df[['clean_text', 'exclamation_count', 'superlative_count']].sample(5)

,clean_text,exclamation_count,superlative_count
33299,granddaughter love thing play several time day,1,1
7785,bought replace smaller pack worked well carrie...,0,1
32595,read every book written author one far favorit...,1,1
9163,took reset system died update review find anot...,0,0
34030,paid little didnt expect much fun first didnt ...,0,0


In [32]:
df.to_csv('../data/processed/features_reviews.csv', index=False)